# 00 — Helpers

Shared constants, utility functions, and Spark schema definitions.
`%run` this notebook from all others before calling any helper.

> **Note:** Install Faker in your Fabric Environment before running: `pip install faker`

In [ ]:
import random
import math
import re
from datetime import date, datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql import types as T

## Company Constants

In [ ]:
BRANCHES = [
    {
        "branch_id": 1, "branch_code": "HF", "branch_name": "Hartfield",
        "is_hq": True, "is_distribution_hub": True,
        "address_line_1": "12 Industrial Way",
        "address_line_2": "Hartfield Industrial Estate",
        "town": "Hartfield", "postcode": "HF1 4RX",
        "phone": "+44 1200 100 200",
        "email": "hartfield@rocklinesupplies.co.uk",
        "manager_name": "Kevin Marsh",
        "opening_mon_fri": "07:00-17:30", "opening_saturday": "07:00-13:00",
        "opening_sunday": None,
    },
    {
        "branch_id": 2, "branch_code": "DR", "branch_name": "Draystone",
        "is_hq": False, "is_distribution_hub": False,
        "address_line_1": "Unit 4 Commerce Park", "address_line_2": None,
        "town": "Draystone", "postcode": "DR3 8WQ",
        "phone": "+44 1200 200 310",
        "email": "draystone@rocklinesupplies.co.uk",
        "manager_name": "Donna Feltham",
        "opening_mon_fri": "07:00-17:00", "opening_saturday": "07:00-12:00",
        "opening_sunday": None,
    },
    {
        "branch_id": 3, "branch_code": "CW", "branch_name": "Crestwick",
        "is_hq": False, "is_distribution_hub": False,
        "address_line_1": "8 Millbrook Road",
        "address_line_2": "Crestwick Trading Estate",
        "town": "Crestwick", "postcode": "CW7 2NL",
        "phone": "+44 1200 300 420",
        "email": "crestwick@rocklinesupplies.co.uk",
        "manager_name": "Stuart Gallagher",
        "opening_mon_fri": "07:30-17:00", "opening_saturday": None,
        "opening_sunday": None,
    },
    {
        "branch_id": 4, "branch_code": "NB", "branch_name": "Norbeck",
        "is_hq": False, "is_distribution_hub": False,
        "address_line_1": "Norbeck Trade Estate, Yard 12", "address_line_2": None,
        "town": "Norbeck", "postcode": "NB4 6PK",
        "phone": "+44 1200 400 530",
        "email": "norbeck@rocklinesupplies.co.uk",
        "manager_name": "Fiona Cartwright",
        "opening_mon_fri": "07:00-17:30", "opening_saturday": "08:00-13:00",
        "opening_sunday": None,
    },
    {
        "branch_id": 5, "branch_code": "LH", "branch_name": "Lyndhurst",
        "is_hq": False, "is_distribution_hub": False,
        "address_line_1": "20 Riverside Business Park", "address_line_2": None,
        "town": "Lyndhurst", "postcode": "LH2 9SJ",
        "phone": "+44 1200 500 640",
        "email": "lyndhurst@rocklinesupplies.co.uk",
        "manager_name": "Chris Alderton",
        "opening_mon_fri": "07:00-17:00", "opening_saturday": "07:30-12:00",
        "opening_sunday": None,
    },
    {
        "branch_id": 6, "branch_code": "PB", "branch_name": "Pembridge",
        "is_hq": False, "is_distribution_hub": False,
        "address_line_1": "Unit 9 Pembridge Gate", "address_line_2": None,
        "town": "Pembridge", "postcode": "PM1 3BF",
        "phone": "+44 1200 600 750",
        "email": "pembridge@rocklinesupplies.co.uk",
        "manager_name": "Rachel Soames",
        "opening_mon_fri": "07:30-17:00", "opening_saturday": "08:00-12:00",
        "opening_sunday": None,
    },
]

CATEGORIES = [
    {"category_id": 1, "category_code": "CC", "category_name": "Concrete & Cement",      "sku_prefix": "CC", "approx_sku_count": 1240},
    {"category_id": 2, "category_code": "ST", "category_name": "Structural Steel",        "sku_prefix": "ST", "approx_sku_count": 980},
    {"category_id": 3, "category_code": "TM", "category_name": "Timber & Sheet",          "sku_prefix": "TM", "approx_sku_count": 1600},
    {"category_id": 4, "category_code": "RF", "category_name": "Roofing",                 "sku_prefix": "RF", "approx_sku_count": 870},
    {"category_id": 5, "category_code": "MA", "category_name": "Masonry",                 "sku_prefix": "MA", "approx_sku_count": 1100},
    {"category_id": 6, "category_code": "PD", "category_name": "Plumbing & Drainage",     "sku_prefix": "PD", "approx_sku_count": 1920},
    {"category_id": 7, "category_code": "EL", "category_name": "Electrical",              "sku_prefix": "EL", "approx_sku_count": 2100},
    {"category_id": 8, "category_code": "FF", "category_name": "Fixings & Fasteners",     "sku_prefix": "FF", "approx_sku_count": 2350},
]

LEADERSHIP = [
    ("Margaret", "Lindqvist", "Chief Executive Officer",   "Management"),
    ("David",    "Okoro",     "Chief Operating Officer",   "Management"),
    ("Sarah",    "Whitmore",  "Finance Director",          "Finance"),
    ("Paul",     "Hendricks", "Head of Procurement",       "Purchasing"),
    ("Aisha",    "Brennan",   "Head of Sales",             "Sales"),
    ("Tom",      "Calloway",  "Head of Logistics",         "Logistics"),
    ("Priya",    "Nair",      "IT & Digital Manager",      "IT"),
    ("James",    "Vickers",   "HR Manager",                "HR"),
]

## Seed Products

Real product data sourced from the Rockline Supplies website.

In [ ]:
SEED_PRODUCTS = [
    # Concrete & Cement (category_id=1)
    {"sku": "CC-0012", "product_name": "General Purpose Portland Cement", "category_id": 1, "unit_of_measure": "bag_25kg",      "pack_size_description": "Pallet (56 bags)",  "standard_cost_gbp": 5.20,   "is_stocked_at_hub_only": False},
    {"sku": "CC-0015", "product_name": "Rapid Set Cement",                "category_id": 1, "unit_of_measure": "bag_25kg",      "pack_size_description": "Pallet (56 bags)",  "standard_cost_gbp": 7.80,   "is_stocked_at_hub_only": False},
    {"sku": "CC-0041", "product_name": "Ready-Mix Concrete C20",          "category_id": 1, "unit_of_measure": "m3",            "pack_size_description": "Min. 3 m3",         "standard_cost_gbp": 82.00,  "is_stocked_at_hub_only": True},
    {"sku": "CC-0042", "product_name": "Ready-Mix Concrete C30",          "category_id": 1, "unit_of_measure": "m3",            "pack_size_description": "Min. 3 m3",         "standard_cost_gbp": 95.00,  "is_stocked_at_hub_only": True},
    {"sku": "CC-0078", "product_name": "Floor Screed Dry Mix",            "category_id": 1, "unit_of_measure": "bag_25kg",      "pack_size_description": "Pallet (48 bags)",  "standard_cost_gbp": 6.40,   "is_stocked_at_hub_only": False},
    {"sku": "CC-0091", "product_name": "Bonding Adhesive Admixture 5L",   "category_id": 1, "unit_of_measure": "each",          "pack_size_description": "Box (4 tubs)",      "standard_cost_gbp": 9.50,   "is_stocked_at_hub_only": False},
    {"sku": "CC-0104", "product_name": "Waterproof Render Mix",           "category_id": 1, "unit_of_measure": "bag_25kg",      "pack_size_description": "Pallet (48 bags)",  "standard_cost_gbp": 8.20,   "is_stocked_at_hub_only": False},
    # Structural Steel (category_id=2)
    {"sku": "ST-0022", "product_name": "Universal Column UC 152x152x23",  "category_id": 2, "unit_of_measure": "linear_metre",  "pack_size_description": "6m or 12m",         "standard_cost_gbp": 28.50,  "is_stocked_at_hub_only": False},
    {"sku": "ST-0031", "product_name": "Universal Beam UB 203x102x23",    "category_id": 2, "unit_of_measure": "linear_metre",  "pack_size_description": "6m or 12m",         "standard_cost_gbp": 22.80,  "is_stocked_at_hub_only": False},
    {"sku": "ST-0045", "product_name": "Square Hollow Section 50x50x3",   "category_id": 2, "unit_of_measure": "linear_metre",  "pack_size_description": "6m",                "standard_cost_gbp": 12.40,  "is_stocked_at_hub_only": False},
    {"sku": "ST-0067", "product_name": "High-Yield Rebar T12",            "category_id": 2, "unit_of_measure": "linear_metre",  "pack_size_description": "Bundle 50",         "standard_cost_gbp": 1.85,   "is_stocked_at_hub_only": False},
    {"sku": "ST-0068", "product_name": "High-Yield Rebar T16",            "category_id": 2, "unit_of_measure": "linear_metre",  "pack_size_description": "Bundle 30",         "standard_cost_gbp": 3.20,   "is_stocked_at_hub_only": False},
    {"sku": "ST-0102", "product_name": "Steel Mesh A393",                 "category_id": 2, "unit_of_measure": "sheet",         "pack_size_description": "2.4 x 4.8 m",      "standard_cost_gbp": 48.00,  "is_stocked_at_hub_only": False},
    # Timber & Sheet (category_id=3)
    {"sku": "TM-0010", "product_name": "C16 Carcassing Timber 47x100mm",  "category_id": 3, "unit_of_measure": "linear_metre",  "pack_size_description": None,                "standard_cost_gbp": 1.20,   "is_stocked_at_hub_only": False},
    {"sku": "TM-0011", "product_name": "C16 Carcassing Timber 47x150mm",  "category_id": 3, "unit_of_measure": "linear_metre",  "pack_size_description": None,                "standard_cost_gbp": 1.80,   "is_stocked_at_hub_only": False},
    {"sku": "TM-0052", "product_name": "OSB3 Flooring Board 18mm",        "category_id": 3, "unit_of_measure": "sheet",         "pack_size_description": "2440x1220mm",       "standard_cost_gbp": 14.50,  "is_stocked_at_hub_only": False},
    {"sku": "TM-0055", "product_name": "Structural Plywood CE2+ 18mm",    "category_id": 3, "unit_of_measure": "sheet",         "pack_size_description": "2440x1220mm",       "standard_cost_gbp": 22.00,  "is_stocked_at_hub_only": False},
    {"sku": "TM-0072", "product_name": "Moisture-Resistant MDF 12mm",     "category_id": 3, "unit_of_measure": "sheet",         "pack_size_description": "2440x1220mm",       "standard_cost_gbp": 18.50,  "is_stocked_at_hub_only": False},
    {"sku": "TM-0088", "product_name": "Treated Fence Post 75x75x1800",   "category_id": 3, "unit_of_measure": "each",          "pack_size_description": None,                "standard_cost_gbp": 6.80,   "is_stocked_at_hub_only": False},
    # Masonry (category_id=5)
    {"sku": "MA-0005", "product_name": "Commons Brick Wirecut",           "category_id": 5, "unit_of_measure": "each",          "pack_size_description": "Pack 500",          "standard_cost_gbp": 0.35,   "is_stocked_at_hub_only": False},
    {"sku": "MA-0012", "product_name": "Facing Brick Red Smooth",         "category_id": 5, "unit_of_measure": "each",          "pack_size_description": "Pack 500",          "standard_cost_gbp": 0.55,   "is_stocked_at_hub_only": False},
    {"sku": "MA-0031", "product_name": "Aircrete Block 7.3N 100mm",       "category_id": 5, "unit_of_measure": "each",          "pack_size_description": "Pallet 72",         "standard_cost_gbp": 1.10,   "is_stocked_at_hub_only": False},
    {"sku": "MA-0044", "product_name": "Dense Concrete Block 7N 140mm",   "category_id": 5, "unit_of_measure": "each",          "pack_size_description": "Pallet 60",         "standard_cost_gbp": 2.20,   "is_stocked_at_hub_only": False},
    {"sku": "MA-0061", "product_name": "Galvanised Wall Tie 225mm",       "category_id": 5, "unit_of_measure": "box",           "pack_size_description": "Box 250",           "standard_cost_gbp": 8.50,   "is_stocked_at_hub_only": False},
    {"sku": "MA-0074", "product_name": "Padstone 440x215x100mm",          "category_id": 5, "unit_of_measure": "each",          "pack_size_description": None,                "standard_cost_gbp": 14.00,  "is_stocked_at_hub_only": False},
]

_SEED_SKUS = {p["sku"] for p in SEED_PRODUCTS}

## Utility Functions

In [ ]:
# ── Product generation helpers ──────────────────────────────────────────

_CC_ADJECTIVES = ["General Purpose", "Rapid Set", "Waterproof", "Self-Levelling",
                  "Fibre-Reinforced", "Sulphate-Resistant", "White", "High-Strength"]
_CC_NOUNS      = ["Portland Cement", "Render Mix", "Floor Screed", "Concrete Mix C25",
                  "Concrete Mix C35", "Mortar Mix", "Grout Mix", "Resin Anchor"]
_CC_UOM        = ["bag_25kg", "bag_25kg", "bag_25kg", "m3", "each"]
_CC_COSTS      = (4.0, 110.0)

_ST_SECTIONS   = ["Universal Beam UB", "Universal Column UC", "Square Hollow Section",
                  "Circular Hollow Section", "Angle Iron", "Parallel Flange Channel"]
_ST_SIZES      = ["100x50", "127x76", "152x89", "178x102", "203x133", "254x146",
                  "305x165", "50x50x5", "60x60x5", "80x80x6", "100x100x8"]
_ST_REBAR      = ["T8", "T10", "T12", "T16", "T20", "T25"]
_ST_UOM        = ["linear_metre", "linear_metre", "sheet", "each"]
_ST_COSTS      = (1.5, 350.0)

_TM_GRADES     = ["C16", "C24", "TR26"]
_TM_SIZES      = ["47x75mm", "47x100mm", "47x125mm", "47x150mm", "47x200mm", "75x150mm"]
_TM_SHEETS     = ["OSB3 11mm", "OSB3 18mm", "OSB3 22mm", "Plywood CE2+ 12mm",
                  "Plywood CE2+ 18mm", "MDF Standard 18mm", "MDF MR 18mm",
                  "Chipboard Flooring 18mm", "Hardboard 3mm"]
_TM_UOM        = ["linear_metre", "linear_metre", "sheet", "each"]
_TM_COSTS      = (0.8, 45.0)

_RF_PRODUCTS   = ["Concrete Roof Tile Plain", "Clay Pantile", "Slate Effect Tile",
                  "Bitumen Felt Underlay", "Breathable Membrane", "Lead Flashing Roll",
                  "EPDM Rubber Membrane", "Fibreglass Gutter 112mm", "uPVC Fascia Board",
                  "Ridge Tile Half Round", "Dry Ridge Kit", "Roofing Batten 25x38mm",
                  "Mineral Felt Shed Roll", "Polycarbonate Sheet Clear 10mm"]
_RF_UOM        = ["each", "each", "roll", "sheet", "linear_metre", "box"]
_RF_COSTS      = (0.45, 160.0)

_MA_BRICKS     = ["Commons Wirecut", "Facing Brick Red Smooth", "Facing Brick Buff",
                  "Engineering Brick Blue", "Engineering Brick Red", "Thermalite Block",
                  "Celcon Block", "Lignacite Block"]
_MA_EXTRAS     = ["Cavity Wall Tie 200mm", "Joist Hanger 47mm", "Restraint Strap 1200mm",
                  "DPC Roll 300mm", "DPC Roll 450mm", "Lintel Galvanised 1200mm",
                  "Lintel Galvanised 1500mm", "Mortar Plasticiser 5L"]
_MA_UOM        = ["each", "each", "box", "roll", "each"]
_MA_COSTS      = (0.30, 125.0)

_PD_PIPES      = ["Copper Pipe 15mm", "Copper Pipe 22mm", "Copper Pipe 28mm",
                  "MDPE Blue 20mm", "MDPE Blue 25mm", "uPVC Soil Pipe 110mm",
                  "ABS Waste Pipe 32mm", "ABS Waste Pipe 40mm", "Clay Drain Pipe 100mm",
                  "Corrugated Drain 60mm", "Polypipe Overflow 21mm"]
_PD_FITTINGS   = ["Copper Elbow 15mm", "Copper Tee 15mm", "Copper Coupling 22mm",
                  "Isolation Valve 15mm", "Ball Valve 22mm", "Gate Valve 28mm",
                  "Channel Drain 100mm", "Inspection Chamber 315mm",
                  "Push-Fit Elbow 32mm", "Boss Pipe Connector 110mm"]
_PD_UOM        = ["linear_metre", "each", "each", "box"]
_PD_COSTS      = (1.80, 180.0)

_EL_CABLES     = ["Twin & Earth 1.5mm2", "Twin & Earth 2.5mm2", "Twin & Earth 6mm2",
                  "SWA Armoured 2.5mm2", "SWA Armoured 6mm2", "Flex 3 Core 1.0mm2",
                  "Single Core 4mm2 Brown", "Fire Alarm Cable 1.5mm2"]
_EL_EXTRAS     = ["Consumer Unit 10-way", "RCD Socket 13A", "Switched Spur 13A",
                  "20mm Steel Conduit 3m", "Plastic Trunking 50x50mm 3m",
                  "Junction Box 5-way IP20", "Cable Clip 4mm White (100)",
                  "Earth Rod 1.2m", "Gland M20", "Armoured Gland M32"]
_EL_UOM        = ["linear_metre", "each", "box", "each"]
_EL_COSTS      = (0.75, 260.0)

_FF_BOLTS      = ["Hex Bolt M6x20", "Hex Bolt M8x30", "Hex Bolt M10x50",
                  "Hex Bolt M12x60", "Hex Bolt M16x80", "Hex Bolt M20x100"]
_FF_SCREWS     = ["Wood Screw 3.5x35mm (200)", "Wood Screw 4.0x50mm (200)",
                  "Drywall Screw 3.5x32mm (500)", "Self-Drill Screw 4.8x19mm (200)",
                  "Coach Screw M8x80mm (50)", "Coach Screw M10x100mm (50)"]
_FF_ANCHORS    = ["Rawlbolt M8 (10)", "Rawlbolt M10 (10)", "Chemical Anchor 300ml",
                  "Frame Anchor 7.5x132mm (50)", "Cavity Toggle M5 (20)",
                  "Nylon Plug 6mm (100)", "Nylon Plug 8mm (100)"]
_FF_UOM        = ["box", "each", "each", "box"]
_FF_COSTS      = (0.01, 55.0)

_CATEGORY_TEMPLATES = {
    1: (_CC_ADJECTIVES, _CC_NOUNS,    _CC_UOM,  _CC_COSTS),
    2: (_ST_SECTIONS,   _ST_SIZES,    _ST_UOM,  _ST_COSTS),
    3: (_TM_GRADES,     _TM_SIZES,    _TM_UOM,  _TM_COSTS),
    4: (_RF_PRODUCTS,   [],           _RF_UOM,  _RF_COSTS),
    5: (_MA_BRICKS,     _MA_EXTRAS,   _MA_UOM,  _MA_COSTS),
    6: (_PD_PIPES,      _PD_FITTINGS, _PD_UOM,  _PD_COSTS),
    7: (_EL_CABLES,     _EL_EXTRAS,   _EL_UOM,  _EL_COSTS),
    8: (_FF_BOLTS,      _FF_SCREWS + _FF_ANCHORS, _FF_UOM, _FF_COSTS),
}

def generate_product_name(cat_id, rng):
    lists_a, lists_b, _, _ = _CATEGORY_TEMPLATES[cat_id]
    if not lists_b:
        return rng.choice(lists_a)
    if cat_id in (1, 3):
        return f"{rng.choice(lists_a)} {rng.choice(lists_b)}"
    if cat_id == 2:
        if rng.random() < 0.3:
            return f"High-Yield Rebar {rng.choice(_ST_REBAR)}"
        return f"{rng.choice(lists_a)} {rng.choice(lists_b)}"
    return rng.choice(lists_a + lists_b)

def generate_product_uom(cat_id, rng):
    _, _, uom_list, _ = _CATEGORY_TEMPLATES[cat_id]
    return rng.choice(uom_list)

def generate_product_cost(cat_id, rng):
    lo, hi = _CATEGORY_TEMPLATES[cat_id][3]
    return round(rng.uniform(lo, hi), 2)

In [ ]:
# ── General helpers ─────────────────────────────────────────────────────

_TOWN_POSTCODES = {
    "Hartfield": "HF1", "Draystone": "DR3", "Crestwick": "CW7",
    "Norbeck": "NB4",  "Lyndhurst": "LH2", "Pembridge": "PM1",
}
_POSTCODE_SUFFIXES = ["2AX", "4BT", "6DW", "8EQ", "3FL", "5GK", "7HN", "9JP",
                      "1KR", "2LS", "4MT", "6NV", "8PY", "3QZ", "5RB"]

def uk_postcode(town, rng):
    prefix = _TOWN_POSTCODES.get(town, f"{town[:2].upper()}{rng.randint(1,9)}")
    return f"{prefix} {rng.choice(_POSTCODE_SUFFIXES)}"

def random_uk_phone(rng):
    area = rng.choice(["1200", "1300", "1400", "1500", "1600", "1700"])
    return f"+44 {area} {rng.randint(100,999)} {rng.randint(100,999)}"

def make_sku(prefix, n):
    return f"{prefix}-{n:04d}"

def trade_email(company_name):
    clean = re.sub(r'[^a-z0-9]', '', company_name.lower().split()[0])
    return f"orders@{clean}.co.uk"

def employee_email(first, last):
    return f"{first.lower()}.{last.lower()}@rocklinesupplies.co.uk"

_SEASONAL_WEIGHTS = {1:0.65, 2:0.70, 3:0.90, 4:1.10, 5:1.25, 6:1.30,
                     7:1.30, 8:1.25, 9:1.15, 10:1.00, 11:0.75, 12:0.60}

def apply_seasonal_weight(d):
    return _SEASONAL_WEIGHTS.get(d.month, 1.0)

def business_day_offset(d, n):
    current = d
    steps = 0
    while steps < n:
        current += timedelta(days=1)
        if current.weekday() < 5:
            steps += 1
    return current

def date_spine(start_date, end_date):
    days = []
    cur = start_date
    while cur <= end_date:
        if cur.weekday() < 5:
            days.append(cur)
        cur += timedelta(days=1)
    return days

def poisson_sample(lam, rng):
    if lam <= 0:
        return 0
    L = math.exp(-min(lam, 500))
    k, p = 0, 1.0
    while p > L:
        k += 1
        p *= rng.random()
    return k - 1

def to_spark_df(rows, schema):
    if not rows:
        return spark.createDataFrame([], schema)
    return spark.createDataFrame(rows, schema)

## Spark Schema Definitions

In [ ]:
from pyspark.sql.types import (StructType, StructField, IntegerType, LongType,
    StringType, BooleanType, DateType, TimestampType, DecimalType, DoubleType)

SCHEMA_DIM_BRANCH = StructType([
    StructField("branch_id",           IntegerType(),  False),
    StructField("branch_code",         StringType(),   False),
    StructField("branch_name",         StringType(),   False),
    StructField("is_hq",               BooleanType(),  False),
    StructField("is_distribution_hub", BooleanType(),  False),
    StructField("address_line_1",      StringType(),   False),
    StructField("address_line_2",      StringType(),   True),
    StructField("town",                StringType(),   False),
    StructField("postcode",            StringType(),   False),
    StructField("phone",               StringType(),   False),
    StructField("email",               StringType(),   False),
    StructField("manager_name",        StringType(),   False),
    StructField("opening_mon_fri",     StringType(),   False),
    StructField("opening_saturday",    StringType(),   True),
    StructField("opening_sunday",      StringType(),   True),
    StructField("created_at",          TimestampType(),False),
])

SCHEMA_DIM_PRODUCT_CATEGORY = StructType([
    StructField("category_id",      IntegerType(), False),
    StructField("category_code",    StringType(),  False),
    StructField("category_name",    StringType(),  False),
    StructField("sku_prefix",       StringType(),  False),
    StructField("approx_sku_count", IntegerType(), False),
])

SCHEMA_DIM_SUPPLIER = StructType([
    StructField("supplier_id",          IntegerType(),       False),
    StructField("supplier_name",        StringType(),        False),
    StructField("contact_name",         StringType(),        False),
    StructField("phone",                StringType(),        False),
    StructField("email",                StringType(),        False),
    StructField("address_line_1",       StringType(),        False),
    StructField("town",                 StringType(),        False),
    StructField("postcode",             StringType(),        False),
    StructField("payment_terms_days",   IntegerType(),       False),
    StructField("lead_time_days",       IntegerType(),       False),
    StructField("category_id",          IntegerType(),       False),
    StructField("is_active",            BooleanType(),       False),
    StructField("created_at",           TimestampType(),     False),
])

SCHEMA_DIM_PRODUCT = StructType([
    StructField("product_id",               IntegerType(),        False),
    StructField("sku",                      StringType(),         False),
    StructField("product_name",             StringType(),         False),
    StructField("category_id",              IntegerType(),        False),
    StructField("unit_of_measure",          StringType(),         False),
    StructField("pack_size_description",    StringType(),         True),
    StructField("standard_cost_gbp",        DecimalType(10,2),    False),
    StructField("list_price_gbp",           DecimalType(10,2),    False),
    StructField("trade_price_gbp",          DecimalType(10,2),    False),
    StructField("weight_kg",                DecimalType(8,2),     True),
    StructField("is_active",                BooleanType(),        False),
    StructField("is_stocked_at_hub_only",   BooleanType(),        False),
    StructField("supplier_id",              IntegerType(),        False),
    StructField("created_at",               TimestampType(),      False),
])

SCHEMA_DIM_CUSTOMER = StructType([
    StructField("customer_id",          IntegerType(),     False),
    StructField("customer_type",        StringType(),      False),
    StructField("company_name",         StringType(),      True),
    StructField("first_name",           StringType(),      False),
    StructField("last_name",            StringType(),      False),
    StructField("email",                StringType(),      False),
    StructField("phone",                StringType(),      False),
    StructField("address_line_1",       StringType(),      False),
    StructField("town",                 StringType(),      False),
    StructField("postcode",             StringType(),      False),
    StructField("account_number",       StringType(),      False),
    StructField("credit_limit_gbp",     DecimalType(10,2), True),
    StructField("payment_terms_days",   IntegerType(),     False),
    StructField("preferred_branch_id",  IntegerType(),     False),
    StructField("account_manager_id",   IntegerType(),     True),
    StructField("is_active",            BooleanType(),     False),
    StructField("created_at",           TimestampType(),   False),
])

SCHEMA_DIM_EMPLOYEE = StructType([
    StructField("employee_id",          IntegerType(),     False),
    StructField("employee_number",      StringType(),      False),
    StructField("first_name",           StringType(),      False),
    StructField("last_name",            StringType(),      False),
    StructField("job_title",            StringType(),      False),
    StructField("department",           StringType(),      False),
    StructField("branch_id",            IntegerType(),     False),
    StructField("hire_date",            DateType(),        False),
    StructField("salary_gbp",           DecimalType(10,2), False),
    StructField("is_active",            BooleanType(),     False),
    StructField("email",                StringType(),      False),
    StructField("manager_employee_id",  IntegerType(),     True),
    StructField("created_at",           TimestampType(),   False),
])

SCHEMA_FACT_INVENTORY_SNAPSHOT = StructType([
    StructField("snapshot_id",       LongType(),        False),
    StructField("product_id",        IntegerType(),     False),
    StructField("branch_id",         IntegerType(),     False),
    StructField("snapshot_date",     DateType(),        False),
    StructField("quantity_on_hand",  DecimalType(12,3), False),
    StructField("quantity_reserved", DecimalType(12,3), False),
    StructField("quantity_available",DecimalType(12,3), False),
    StructField("reorder_point",     DecimalType(12,3), False),
    StructField("max_stock_level",   DecimalType(12,3), False),
    StructField("average_cost_gbp",  DecimalType(10,4), False),
    StructField("created_at",        TimestampType(),   False),
])

SCHEMA_FACT_INVENTORY_MOVEMENT = StructType([
    StructField("movement_id",     LongType(),        False),
    StructField("product_id",      IntegerType(),     False),
    StructField("branch_id",       IntegerType(),     False),
    StructField("movement_date",   TimestampType(),   False),
    StructField("movement_type",   StringType(),      False),
    StructField("reference_id",    LongType(),        True),
    StructField("reference_type",  StringType(),      True),
    StructField("quantity_change", DecimalType(12,3), False),
    StructField("unit_cost_gbp",   DecimalType(10,4), False),
    StructField("created_at",      TimestampType(),   False),
])

SCHEMA_FACT_SALES_ORDER = StructType([
    StructField("sales_order_id",           LongType(),        False),
    StructField("order_number",             StringType(),      False),
    StructField("customer_id",              IntegerType(),     False),
    StructField("branch_id",               IntegerType(),     False),
    StructField("taken_by_employee_id",     IntegerType(),     False),
    StructField("order_date",               TimestampType(),   False),
    StructField("requested_delivery_date",  DateType(),        True),
    StructField("actual_despatch_date",     DateType(),        True),
    StructField("actual_delivery_date",     DateType(),        True),
    StructField("order_status",             StringType(),      False),
    StructField("fulfilment_type",          StringType(),      False),
    StructField("delivery_address_line_1",  StringType(),      True),
    StructField("delivery_town",            StringType(),      True),
    StructField("delivery_postcode",        StringType(),      True),
    StructField("subtotal_gbp",             DecimalType(12,2), False),
    StructField("vat_gbp",                  DecimalType(12,2), False),
    StructField("total_gbp",               DecimalType(12,2), False),
    StructField("discount_pct",             DecimalType(5,2),  False),
    StructField("is_credit_sale",           BooleanType(),     False),
    StructField("payment_due_date",         DateType(),        True),
    StructField("payment_received_date",    DateType(),        True),
    StructField("created_at",               TimestampType(),   False),
])

SCHEMA_FACT_SALES_ORDER_LINE = StructType([
    StructField("line_id",           LongType(),        False),
    StructField("sales_order_id",    LongType(),        False),
    StructField("line_number",       IntegerType(),     False),
    StructField("product_id",        IntegerType(),     False),
    StructField("quantity",          DecimalType(12,3), False),
    StructField("unit_price_gbp",    DecimalType(10,4), False),
    StructField("line_net_gbp",      DecimalType(12,2), False),
    StructField("line_vat_gbp",      DecimalType(12,2), False),
    StructField("line_total_gbp",    DecimalType(12,2), False),
    StructField("cost_of_goods_gbp", DecimalType(12,2), False),
    StructField("created_at",        TimestampType(),   False),
])

SCHEMA_FACT_PURCHASE_ORDER = StructType([
    StructField("purchase_order_id",        LongType(),        False),
    StructField("po_number",                StringType(),      False),
    StructField("supplier_id",              IntegerType(),     False),
    StructField("branch_id",                IntegerType(),     False),
    StructField("raised_by_employee_id",    IntegerType(),     False),
    StructField("order_date",               DateType(),        False),
    StructField("expected_delivery_date",   DateType(),        False),
    StructField("actual_delivery_date",     DateType(),        True),
    StructField("po_status",                StringType(),      False),
    StructField("subtotal_gbp",             DecimalType(12,2), False),
    StructField("vat_gbp",                  DecimalType(12,2), False),
    StructField("total_gbp",                DecimalType(12,2), False),
    StructField("created_at",               TimestampType(),   False),
])

SCHEMA_FACT_PURCHASE_ORDER_LINE = StructType([
    StructField("line_id",              LongType(),        False),
    StructField("purchase_order_id",    LongType(),        False),
    StructField("line_number",          IntegerType(),     False),
    StructField("product_id",           IntegerType(),     False),
    StructField("quantity_ordered",     DecimalType(12,3), False),
    StructField("quantity_received",    DecimalType(12,3), False),
    StructField("unit_cost_gbp",        DecimalType(10,4), False),
    StructField("line_net_gbp",         DecimalType(12,2), False),
    StructField("line_vat_gbp",         DecimalType(12,2), False),
    StructField("line_total_gbp",       DecimalType(12,2), False),
    StructField("created_at",           TimestampType(),   False),
])

print("Helpers loaded: BRANCHES, CATEGORIES, SEED_PRODUCTS, schemas, utility functions")